# Questão 1

### 1. a)

Queremos uma reflexão que leve $x$ até $||x||e_1$. Ou seja, queremos que $v$ seja o vetor normal ao hiperplano que reflete $x$ em $||x||e_1$. Para $v$ ser o vetor normal a esse hiperplano, ele deve ser a normal a mediatriz entre os vetores $x$ e $||x||e_1$, então $v = x - ||x||e_1$.

Encontrando $\beta$: $$Q_{v}x = x - \beta vv^{*}x = ||x||e_1$$
$$\beta vv^{*}x = x - ||x||e_1 = v$$
$$\beta = \frac{1}{v^{*}x}$$

Abrindo $v^{*}$ temos: $v^{*} = (x - ||x||e_1)^{*} = x^* - ||x||e_{1}^*$

Substituindo: $$\beta = \frac{1}{(x^* - ||x||e_{1}^*)x} = \frac{1}{x^{*}x - ||x||e_{1}^{*}x} = \frac{1}{||x||^2 - ||x||x_1} = \frac{1}{||x||(||x|| - x_1)} = \frac{1}{||x||(||x|| - x_1)} \cdot \frac{||x|| + x_1}{||x|| + x_1} = \frac{||x|| + x_1}{||x||(||x||^2 - x_{1}^2)}$$

Observe que $||x||^2 = x_{1}^2 + x_{2}^2 + \cdots + x_{n}^2$ 

Então $||x||^2 - x_{1}^2 = x_{2}^2 + x_{3}^2 + \cdots + x_{n}^2 = ||y||^2$, já que $y = (x_2, ..., x_n)$

Logo, $$ \frac{||x|| + x_1}{||x||(||x||^2 - x_{1}^2)} = \frac{||x|| + x_1}{||x||\cdot ||y||^2}$$

Observação: note que estamos usando a norma $2$ (euclidiana).

### 1. b)

Temos duas fórmulas equivalentes para $\beta$: $$\beta _{1} = \frac{1}{||x||(||x|| - x_1)} \hspace{4cm} \beta _{2} = \frac{||x|| + x_1}{||x|| \cdot ||y||^2}$$ 
Quando $x_1 > 0$: 

Se $x_1 > 0$ e próximo de $||x||$ temos: $$||x|| - x_1 \approx 0$$
Na fórmula $\beta _{1}$ o denominador envolve diretamente essa subtração, o que pode causar um erro muito grande ao realizar a subtração entre dois float64, perdendo a precisão da resposta. Já $\beta _{2}$ não tem esse problema pois o numerador é a soma de dois positivos e o denominador também não sofre cancelamento.

Quando $x_1 < 0$: 

Se $x_1 < 0$ e próximo de $-||x||$ temos: $$||x|| + x_1 \approx 0$$
Na fórmula $\beta _{2}$ o numerador sofre cancelamento, perdendo a precisão da resposta. Já $\beta _{1}$ não tem esse problema o denominador se transformará em uma soma de números positivos. 

**Mas a pergunta é: por que ao realizar a subtração de dois números muito próximos usando ponto flutuante pode gerar um erro enorme?**

A resposta está em como números de ponto flutuante são armazenados. Ele é dividido em $1$ bit para o sinal, $11$ bits para o campo do expoente e $52$ bits para a mantissa, o que confere uma precisão relativa da ordem de $\epsilon _{maq} \approx 2^{-52} \approx 2.2 \times 10^{-16}$ Isso significa que cada número carrega um erro absoluto da ordem de $\epsilon _{maq} \cdot |x|$. Quando subtraimos dois números muito próximos, os algarismos significativos se cancelam e o que resta no resultado são os bits menos significativos, os mais contaminados por erros de arredondamento (ao armazenar um número em ponto flutuante, o arredondamento para o representável mais próximo afeta a posição menos significativa da mantissa, pois os bits mais significativos já capturam a maior parte do valor com exatidão, e o resíduo que não cabe na representação é arredondado no último bit. Assim, quando a subtração cancela os bits mais significativos, o que sobra no resultado são exatamente os bits que carregavam esse erro de arredondamento). O erro relativo do resultado é da ordem de $\frac{\epsilon _{maq} \cdot ||x||}{||x|| - x_1}$, que explode quando $||x|| - x_1 \rightarrow 0$. 

Portanto, dependendo do sinal de $x_1$, umas das fórmulas sempre é melhor de se usar do que a outra por não envolver cancelamento numérico.


### 1. c)

In [17]:
import numpy as np

def reflector(x):
    m = int(x.shape[0])
    e1 = np.zeros(m, dtype=x.dtype)
    e1[0] = 1 # vetor canonico com 1 na posição 1

    v = x - np.linalg.norm(x).astype(x.dtype) * e1
    y = x[1:]

    if(x[0] > 0): # para diminuir a perda de precisão
        beta = (np.linalg.norm(x) + x[0]) / (np.linalg.norm(x) * (np.linalg.norm(y)**2))
    else:
        beta = 1 / (np.linalg.norm(x) * (np.linalg.norm(x) - x[0]))

    Q = np.eye(m, m) - beta * (v @ v.T)

    return v, beta

x0 = np.array([2, 2], dtype=float)
v0, beta0 = reflector(x0)
print(v0, beta0)


[-0.82842712  2.        ] 0.4267766952966368


### 1. d)

$$v = x - ||x||e_1$$
$$\frac{\partial v}{\partial x} = \frac{\partial x}{\partial x} - e_1\frac{\partial ||x||}{\partial x}$$
$$\frac{\partial v}{\partial x} = I - e_1\frac{\partial ||x||}{\partial x}$$
Utilizando a regra da cadeia $\frac{d}{du}\sqrt{u} = \frac{1}{2\sqrt{u}}u'$:
$$= \frac{1}{2\sqrt{x_{1}^2 + x_{2}^2 + \cdots + x_{n}^2}} \cdot 2x_j = \frac{x_j}{||x||}$$
Juntando todas as componentes:$$\nabla x = \begin{bmatrix}
\frac{x_1}{||x||} \\
\vdots \\
\frac{x_n}{||x||} 
\end{bmatrix} = \frac{x}{||x||}$$
Então, concluimos que: $$J = I - e_1 \frac{x^T}{||x||} = I - \frac{e_1 \cdot x^*}{||x||}$$
Para calcular o número de condicionamento absoluto de de $v$ em relação a $x$ precisamos calcular: $$\kappa _{abs} = ||J||_2 = ||I - \frac{e_1 x^*}{||x||}||_2$$ e como sabemos, a norma 2 de uma matriz é o seu maior valor singular.

Sem perda de generalidade podemos assumirr $||x|| = 1$ (se não for é só normalizar). Então $J = I - e_{1}x^*$.

**O que J faz com qualquer vetor $u$?**

$$Ju = u - e_1(x^{*}u)$$
Note que $x^*u$ é um escalar. Chamando ele de $\alpha$: $$Ju = u - e_1\alpha$$
Ou seja, $J$ só altera a primeira posição do vetor $u$, subtraindo $\alpha$.

Pela desigualdade triangular temos: $$||Ju|| = ||u - \alpha e_1|| \leq ||u|| + |\alpha| ||e_1||$$
Como $|\alpha| = |x^*u| \leq ||x||\cdot||u|| = ||u||$ e $||e_1|| = 1$: $$||Ju|| \leq ||u|| + ||u|| = 2||u||$$ 
Como a definição formal da norma $2$ de uma matriz é: $$||J||_2 = \underset{||u||=1}{max}||Ju||_2$$
Portanto: $$ ||J||_2 = \underset{||u||=1}{max}||Ju|| \leq 2||u|| = 2$$
$$\kappa _{abs} \leq 2$$

**Quando a igualdade é atingida?**

Quando $u = e_1$ e $x = -e_1$:
$$Ju = e_1 - (-1)e_1 = 2e_1 \implies ||Ju|| = 2$$

### 1. e)


In [37]:
print("===================== E X E M P L O  2 x 1 ========================")

x = np.array([2, 2], dtype=np.float32)
v1, beta1 = reflector(x)
print("--------------FLOAT32---------------")
print(f"v: type {v1.dtype}, valor: {v1}")
print(f"beta: type {beta1.dtype}, valor: {beta1}")

y = np.array([2, 2], dtype=np.float64)
v2, beta2 = reflector(y)
print("--------------FLOAT64---------------")
print(f"v: type {v2.dtype}, valor: {v2}")
print(f"beta: type {beta2.dtype}, valor: {beta2}")

print("===================== E X E M P L O  500 x 1 ========================")

y1 = (np.random.rand(500)).astype(np.float64)

x1 = y1.astype(np.float32)
v3, beta3 = reflector(x1)
print("--------------FLOAT32---------------")
print(f"v: type {v3.dtype}")
print(f"beta: type {beta3.dtype}, valor: {beta3}")

v4, beta4 = reflector(y1)
print("--------------FLOAT64---------------")
print(f"v: type {v4.dtype}")
print(f"beta: type {beta4.dtype}, valor: {beta4}")

===================== E X E M P L O  2 x 1 ========================
--------------FLOAT32---------------
v: type float32, valor: [-0.8284271  2.       ]
beta: type float32, valor: 0.4267767071723938
--------------FLOAT64---------------
v: type float64, valor: [-0.82842712  2.        ]
beta: type float64, valor: 0.4267766952966368
===================== E X E M P L O  500 x 1 ========================
--------------FLOAT32---------------
v: type float32
beta: type float32, valor: 0.006217132788151503
--------------FLOAT64---------------
v: type float64
beta: type float64, valor: 0.006217132395303152


Primeiramente podemos ver que em ambos os casos de teste quando eu passo um vetor $Float32$, eu recebo um vetor $Float32$ de resposta, e a mesma coisa vale para um vetor $Float64$.

Note que em ambos os casos, tanto no vetor $2\times1$ quanto no vetor $500\times1$, os valores de $\beta$ começam a se diferenciar praticamente na mesma casa decimal para diferentes $Float$ ($10^{-8}$ para o vetor $\in \mathbb{R}^2$ e $10^{-9}$ para o vetor $\in \mathbb{R}^{500}$). Isso nos indica que o erro não explodirá quando o tamanho do problema crescer (pelo menos de 2 para 500 elementos). Isso também mostra que o algoritmo é estável, mesmo aumentando a dimensão do vetor para 500 elementos (o que envolve acumular centenas de somas e quadrados no cálculo da norma), a diferença entre o $Float32$ e o $Float64$ permaneceu controlada.



### 1. f)

Temos $v = (v_1, v_2, \dots, v_n)$ com $v_1 = x_1 - ||x||$ e com $v_i = x_i$ para $i = 2, \dots, n$. Isso porque quando fazemos $v = x - ||x||e_1$, note que o vetor $v$ difere apenas a primeira posição em relação ao vetor $x$. 

Agora, precisamos expressar $||x||$ e $x_1$ em função de $v$. Perceba que $||y||^2 = x_{2}^2 + \dots + x_{n}^2 = v_{2}^2 + \dots + v_{n}^2 = ||v||^2 - v_{1}^2$.

Sabemos que $||x|| = x_1 - v_1$ e que $||x||^2 = x_{1}^2 + ||y|||^2$, substituindo: $$(x_1 - v_1)^2 = x_{1}^2 + ||y||^2$$
$$x_{1}^2 - 2x_{1}v_{1} + v_{1}^2 = x_{1}^2 + ||y||^2$$
$$x_{1} = \frac{v_{1}^2 - ||y||^2}{2v_1}$$

Como $||x|| = x_1 - v_1$, então: $$||x|| = \frac{v_{1}^2 - ||y||^2}{2v_1} - \frac{2v_{1}^2}{2v_{1}} = \frac{-v_{1}^2 - ||y||^2}{2v_1} = - \frac{||v||^2}{2v_1}$$

Substituindo na fórmula $\beta _{2} = \frac{||x|| + x_1}{||x|| \cdot ||y||^2}$, temos: $$||x|| + x_1 = -\frac{||v||^2}{2v_1} + \frac{v_{1}^2 - ||y||^2}{2v_1} = \frac{-2||y||^2}{2v_1} = \frac{-||y||^2}{v_1}$$

Portanto, $$\beta = \frac{||x|| + x_1}{||x|| \cdot ||y||^2} = \frac{\frac{-||y||^2}{v_1}}{-\frac{||v||^2}{2v_1}(||y||^2)} = \frac{2}{||v||^2}$$

In [44]:
def calc_beta(v):
    return 2 / (np.linalg.norm(v))**2

v = np.array([2*(1 - 2**(1/2)),2], dtype=float)
print(calc_beta(v))

0.42677669529663687


Note que o $\beta$ é praticamente o mesmo da última questão, com a diferença sendo na casa de $10^{-8}$.